### Imports

In [ ]:
import os

import matplotlib.pyplot as plt
import seaborn as sns

from data_processing import clean_data, load_data


### Load data

In [ ]:
df_hv, df_vm = load_data()
df_vm_clean = clean_data(df_vm)

### Info
#### Df hypervisor

In [ ]:
print("--- Summary DataFrame ---")
df_hv.head()
display(df_hv[["cpu_usage", "memory_total", "memory_usage", "num_instances"]].describe())

### Df vm

In [ ]:
print("\n--- Vm original DataFrame ---")
display(df_vm.head())
display(df_vm.shape, len(df_vm["id"].unique()))
display(df_vm[["mem_alloc", "mem_usage", "vcpus", "cpu_usage"]].describe())

print("\n--- Vm clean DataFrame ---")
display(df_vm_clean.head())
display(df_vm_clean.shape, len(df_vm["id"].unique()))
display(df_vm_clean[["mem_alloc", "mem_usage_ratio", "vcpus", "cpu_usage"]].describe())

In [ ]:
s = df_vm["vcpus"].unique()
s.sort()
s

### Histograms

In [ ]:
def cpu_density_plot( 
    vcpus: list[int], 
    x_min: float|None = None, 
    x_max: float|None = None
):
    plt.figure()
    for v in vcpus:
        sns.kdeplot(
            data=df_vm_clean[df_vm_clean["vcpus"] == v],
            x="cpu_usage",
            label=f"{v} vCPU",
            # fill=True,
            alpha=0.9,
        )
            
    plt.xlim(left=x_min, right=x_max)
    plt.legend()
    plt.show()

cpu_density_plot([1,2,4,8,16], 0, 0.35)

In [ ]:
def plot_mean_cpu_timeseries(vcpu_list:list[int]):
    plt.figure(figsize=(12, 6))

    all_vcpus = sorted(df_vm_clean["vcpus"].unique())
    all_vcpus = [x for x in all_vcpus if x in vcpu_list]
    sorted_vcpus = [str(x) for x in all_vcpus]

    sns.lineplot(
        data=df_vm_clean,
        x="timestamp",
        y="cpu_usage",
        hue=df_vm_clean["vcpus"].astype(str),
        hue_order=sorted_vcpus,
        estimator="mean",
        errorbar=None,
        alpha=0.4
    )
    
    plt.title("Mean CPU Usage Timeseries Grouped by vCPUs")
    plt.xlabel("Time")
    plt.ylabel("Mean CPU Usage")
    plt.legend(title="vCPUs")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_mean_cpu_timeseries([8, 16])

In [ ]:
def cpu_density_plot_all(x_min: float | None = None, x_max: float | None = None):
    plt.figure()
    sns.kdeplot(
        data=df_vm_clean,
        x="cpu_usage",
        label="All VMs",
        fill=True,
        alpha=0.9,
    )

    plt.xlim(left=x_min, right=x_max)
    plt.legend()
    plt.show()

cpu_density_plot_all(0, 0.3)

In [ ]:
a = (df_vm_clean["cpu_usage"] < 0.05).mean()
print(f"Less than 0.05 cpu usage: {a * 100:.2f}%")

### Count distribution

In [ ]:
def name_distribution_plot():
    os.makedirs("distr", exist_ok=True)

    group_lengths = df_vm.groupby("id").size()
    print(group_lengths.describe())

    plt.figure(figsize=(10, 6))
    plt.hist(group_lengths, bins=30, edgecolor="black", alpha=0.7)
    plt.title("Group size distribution")
    plt.xlabel("Rows per group")
    plt.ylabel("No of groups")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.savefig("distr/d0")
    plt.show()

    # only for few values
    group_lengths_filtered = group_lengths[group_lengths <= 50]

    # new stats
    print(group_lengths_filtered.describe())
    plt.figure(figsize=(10, 6))
    plt.hist(group_lengths_filtered, bins=50, edgecolor="black", alpha=0.7)
    plt.title("Group size distribution")
    plt.xlabel("Rows per group")
    plt.ylabel("No of groups")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.savefig("distr/d1")
    plt.show()

name_distribution_plot()

#### Find number of series with count larger than threshold

In [ ]:
def find_above(threshold:int):
    os.makedirs("distr", exist_ok=True)
    group_lengths = df_vm.groupby("id").size()
    group_lengths_filtered = group_lengths[group_lengths >= threshold]
    glfs = group_lengths_filtered.sort_values()
    print(f"Count > {threshold} = {len(glfs)}")
    glfs.to_csv("distr/glfs.csv")

find_above(100)